Inputs: Training Set (Data, Labels)
Outputs: Metrics: Loss and Accuracy

In [ ]:
# sync with github so my imports are here
!git clone https://github.com/ellylai/10707-project.git
%cd 10707-project

!pip install -q transformers datasets scikit-learn

import sys
sys.path.append("/content/10707-project")

In [ ]:
# imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

from datasets.download_datasets import *
from utils.training_utils import *
from datasets.create_dataset_splits import *
from datasets.get_waveform_datasets import *

In [ ]:
# wandb and huggingface setup
import wandb
from huggingface_hub import HfApi, login

# Initialize W&B
wandb.init(
    project="audio-deepfake-detection",
    config={
        "learning_rate": lr,
        "architecture": "W2V2_AASIST",
        "dataset": "SpeechFake",
        "epochs": epochs,
    }
)

# Login to HF (Run this once or use a token)
login() 
api = HfApi()
repo_id = "Joel-10707-Project-S26/baseline-w2v2aasist"

In [ ]:
# download data from S3 bucket
# S3 bucket is named s3://10707-project/10707-project/
# files should be named train_dataset, val_dataset, and test_dataset (AudioDataset)
!aws configure
!aws s3 sync s3://10707-project/10707-project/ ./data/
# ./data/ has the structure:
# data/
#   speechfake_splits.csv
#   audio/
#     file1.wav
#     file2.wav
# around 5gb of data
CSV_PATH = "data/speechfake_splits.csv"

train_dataset = AudioDatasetFromCSV(CSV_PATH, split="train")
val_dataset   = AudioDatasetFromCSV(CSV_PATH, split="val")
test_dataset  = AudioDatasetFromCSV(CSV_PATH, split="test")

fatal error: An error occurred (AccessDenied) when calling the ListObjectsV2 operation: Access Denied


NameError: name 'AudioDatasetFromCSV' is not defined

In [ ]:
# args/configs
d_args = {
    "filts": [[1, 32], [32, 32], [32, 64], [64, 64], [64, 128]], # Example AASIST filter bank
    "gat_dims": [64, 32],
    "pool_ratios": [0.5, 0.7, 0.5],
    "temperatures": [2.0, 2.0, 1.0],
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 8  # Keep small for W2V2
lr = 0.0001
epochs = 10

In [ ]:
# import model and initialize optimizer
from baseline.w2v2_aasist import W2V2_AASIST

model = W2V2_AASIST(d_args).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

In [ ]:
# train
for epoch in range(epochs):
    loss = train_epoch(model, train_loader, optimizer, criterion, device)
    loss, acc = validate(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}: Loss {loss:.4f}, Val Acc {acc:.4f}")